# MUSAC LAS Classifier — Training and Inference on Google Colab

This notebook runs the full pipeline on Colab GPU:

1. Install the package from GitHub
2. Upload/mount STPLS3D data
3. Prepare training data (tile, encode, generate images)
4. Train the U-Net segmentation model with full metric collection
5. Visualize training curves and confusion matrices
6. Evaluate with best model
7. Run inference pipeline with timing
8. Download results

**GPU required**: Go to Runtime → Change runtime type → T4 GPU

## 1. Setup & Installation

In [ ]:
# Check GPU availability
!nvidia-smi

# Install the package from GitHub
!pip install git+https://github.com/AlexeyKozhakin/lidar-visual-interface.git

# Install training dependencies
!pip install matplotlib pandas scikit-learn psutil gdown

# Verify installation
import musac_las_classifier
print(f"musac_las_classifier version: {musac_las_classifier.__version__}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive (for data upload/download)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# If your STPLS3D data is on Google Drive, set the path here:
# DRIVE_DATA_DIR = "/content/drive/MyDrive/stpls3d"

## 3. Prepare Data

Upload your STPLS3D raw LAS files to the `raw/` directory, or copy from Google Drive.

In [ ]:
import os

# Directory structure
DATA_DIR = "/content/data/stpls3d"
RAW_DIR = os.path.join(DATA_DIR, "raw")
os.makedirs(RAW_DIR, exist_ok=True)

# Option A: Copy from Google Drive
# !cp /content/drive/MyDrive/stpls3d/*.las {RAW_DIR}/

# Option B: Upload files directly
# from google.colab import files
# uploaded = files.upload()  # Upload .las files
# for fname in uploaded:
#     !mv {fname} {RAW_DIR}/

# Check files
print("Raw LAS files:")
!ls -lh {RAW_DIR}/

In [ ]:
# Run data preparation pipeline: raw LAS -> tiles -> tensors -> images
!python -m training.prepare_data \
    --raw-dir {RAW_DIR} \
    --output-dir {DATA_DIR} \
    --tile-size 250 \
    --num-points 30000 \
    --grid-size 512 \
    --k-nn 4

# Verify outputs
features_dir = os.path.join(DATA_DIR, "img_features")
masks_dir = os.path.join(DATA_DIR, "img_class")
print(f"\nFeature images: {len(os.listdir(features_dir))}")
print(f"Mask images: {len(os.listdir(masks_dir))}")

## 4. Train the Model

Training with full metric collection: mIoU, best model selection, confusion matrices, loss curves.

In [ ]:
CHECKPOINT_DIR = "/content/checkpoints"

!python -m training.train \
    --features-dir {features_dir} \
    --masks-dir {masks_dir} \
    --epochs 100 \
    --batch-size 8 \
    --lr 1e-3 \
    --device auto \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --num-classes 20 \
    --seed 42

## 5. Visualize Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load epoch summary
epoch_df = pd.read_csv(os.path.join(CHECKPOINT_DIR, "epoch_summary.csv"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(epoch_df["epoch"], epoch_df["train_loss"], label="Train")
axes[0].plot(epoch_df["epoch"], epoch_df["val_loss"], label="Validation")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Dice Loss")
axes[0].set_title("Training Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# mIoU curves
axes[1].plot(epoch_df["epoch"], epoch_df["train_miou"], label="Train")
axes[1].plot(epoch_df["epoch"], epoch_df["val_miou"], label="Validation")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("mIoU")
axes[1].set_title("Mean IoU")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print best epoch
best_idx = epoch_df["val_miou"].idxmax()
print(f"Best epoch: {int(epoch_df.loc[best_idx, 'epoch'])} "
      f"(val mIoU: {epoch_df.loc[best_idx, 'val_miou']:.4f})")

# Print timing
print(f"\nAvg epoch time: {epoch_df['epoch_time_sec'].mean():.1f}s")
print(f"Total training time: {epoch_df['epoch_time_sec'].sum() / 60:.1f} min")

## 6. Per-Class Metrics & Confusion Matrix

In [ ]:
from musac_las_classifier.constants import CLASS_NAMES

# Load per-class metrics from the last epoch
metrics_df = pd.read_csv(os.path.join(CHECKPOINT_DIR, "metrics_log.csv"))
last_epoch = metrics_df["epoch"].max()
val_final = metrics_df[
    (metrics_df["dataset"] == "Val") & (metrics_df["epoch"] == last_epoch)
]

print(f"{'ID':>3} {'Class':<20} {'Prec':>8} {'Recall':>8} {'IoU':>8} {'F1':>8}")
print("-" * 60)
for _, row in val_final.iterrows():
    cls = int(row["class"])
    name = CLASS_NAMES.get(cls, f"Class {cls}")
    print(f"{cls:>3} {name:<20} {row['precision']:>8.4f} {row['recall']:>8.4f} "
          f"{row['iou']:>8.4f} {row['f1']:>8.4f}")
print(f"\nmIoU: {val_final['iou'].mean():.4f}")

In [ ]:
from IPython.display import Image, display

# Display confusion matrix
cm_path = os.path.join(CHECKPOINT_DIR, "confusion_matrices", "confusion_matrix_val_final.png")
if os.path.exists(cm_path):
    display(Image(filename=cm_path))
else:
    print("Confusion matrix image not found")

## 7. Evaluate Best Model

In [ ]:
EVAL_DIR = "/content/eval_results"

!python -m training.evaluate \
    --checkpoint {os.path.join(CHECKPOINT_DIR, "best_model.pth")} \
    --features-dir {features_dir} \
    --masks-dir {masks_dir} \
    --batch-size 8 \
    --device auto \
    --num-classes 20 \
    --seed 42 \
    --output-dir {EVAL_DIR}

In [ ]:
import json

# Display evaluation summary
with open(os.path.join(EVAL_DIR, "evaluation_summary.json")) as f:
    summary = json.load(f)
print(json.dumps(summary, indent=2))

# Display confusion matrix
cm_eval_path = os.path.join(EVAL_DIR, "confusion_matrix_eval.png")
if os.path.exists(cm_eval_path):
    display(Image(filename=cm_eval_path))

## 8. Run Inference Pipeline with Timing

Run the full inference pipeline on a test scene and measure per-stage performance.

In [ ]:
from musac_las_classifier import LasClassificationPipeline, PipelineConfig

# Download ResNet34 encoder weights
!python -m scripts.download_models

config = PipelineConfig.for_multiclass(
    checkpoint_path=os.path.join(CHECKPOINT_DIR, "best_model.pth"),
    encoder_weights_path="models/resnet34-333f7ec4.pth",
    device="auto",
)

pipeline = LasClassificationPipeline(config, workdir="/content/workdir")

# Use raw data or a specific test scene
# Adjust the path to your test LAS files
pipeline.load_las(RAW_DIR)
pipeline.run_classification("/content/output/classified.las")

# Print timing report
report = pipeline.get_timing_report()
print("\n=== Timing Report ===")
print(json.dumps(report, indent=2))

## 9. Download Results

In [ ]:
import shutil
from google.colab import files

# Zip training results
shutil.make_archive("/content/training_results", "zip", CHECKPOINT_DIR)
print("Training results zipped.")

# Zip evaluation results
if os.path.exists(EVAL_DIR):
    shutil.make_archive("/content/eval_results_archive", "zip", EVAL_DIR)
    print("Evaluation results zipped.")

# Download
files.download("/content/training_results.zip")
files.download("/content/eval_results_archive.zip")

# Optionally download the classified LAS and timing report
if os.path.exists("/content/output/classified.las"):
    files.download("/content/output/classified.las")
if os.path.exists("/content/workdir/timing_report.json"):
    files.download("/content/workdir/timing_report.json")

## Hardware Info

Display collected hardware information for the paper.

In [ ]:
hw_path = os.path.join(CHECKPOINT_DIR, "hardware_info.json")
if os.path.exists(hw_path):
    with open(hw_path) as f:
        hw = json.load(f)
    print("Hardware Info:")
    for k, v in hw.items():
        print(f"  {k}: {v}")
else:
    print("Hardware info not found. Run training first.")